# The Ocean Cleanup System 03: Multi-Criteria Decision Analysis (MCDA)

This notebook contains the Multi-Criteria Decision Analysis (MCDA) for our project on The Ocean Cleanup's System 03, using **Preference Function Modeling (PFM)**. It evaluates four alternative concepts for removing plastic from the Great Pacific Garbage Patch, based on how five stakeholder groups rate each alternative on their own criteria. These individual ratings are then aggregated into one overall preference score per alternative.

The notebook is adapted from the course example `EXAMPLE_Gold_Coast_MCDA.ipynb`. The code is unchanged; only the input data and the descriptions are specific to our project.

## Why Preference Function Modeling?

Many decision-making methods used in practice have a flawed mathematical foundation. They combine quantities expressed in different units (e.g. euros, kilograms and square kilometres), mix ordinal data (1st, 2nd, 3rd) with cardinal data (€1 million, 50 t), or apply operations to scales for which they are not defined. Ranking ecological impact from 1 (best) to 4 (worst), for example, does not mean that the alternative ranked 2 is twice as good as the one ranked 4.

This matters for our case because the criteria are a mixture of quantitative measures (plastic removal in kg per time, cost per kg removed) and qualitative assessments (scalability, gear conflict risk, long-term ecosystem recovery). Addition, averaging and weighted summation are only meaningful when the underlying scales have the required measurement properties.

PFM solves this by first expressing every criterion on a common preference scale and only then aggregating. Each stakeholder rates the alternatives per criterion on a scale from **0** (worst outcome for that stakeholder) to **100** (best outcome for that stakeholder). The ratings are then combined with **affine (a-fine) aggregation**, which preserves the properties of the preference scale.

All criteria are formulated so that a higher score is better from the stakeholder's point of view. For a criterion such as *Gear conflict / entanglement risk*, a high score therefore means a low risk.

## What this notebook does

- Load and check the stakeholder ratings and criteria weights.
- Aggregate the criteria ratings into one preference score per stakeholder (Level 1).
- Aggregate the stakeholder scores into one group preference score per alternative (Level 2).

## Input data

All input is read from `MCDA_input/MCDA_Ratings_OceanCleanup_System03.csv`. Its contents are the same as the sheet *Stakeholder Matrix* in `Ocean_Cleanup_Stakeholder_Matrix.xlsx`. To change a rating, a weight, a criterion or an alternative, edit the `.csv` file and keep its structure; the notebook itself does not need to change.

# Problem

The Great Pacific Garbage Patch (GPGP) is an accumulation of floating marine debris in the central North Pacific Ocean. It was predicted as early as 1988 and has since grown to about 1.6 million km², roughly twice the size of Texas. The Ocean Cleanup currently removes plastic from the patch with System 03, a 2.2 km long U-shaped floating barrier that is towed through the patch by two vessels.

To see how System 03 compares with other ways of cleaning up the patch, four alternatives are considered:

| Alternative | Concept |
|---|---|
| **Modular Fleet** | Many smaller, passively drifting collection units that can be added one by one. |
| **Anchored Barrier Array** | Fixed barriers moored in the deep ocean, which rely on the currents to bring the plastic to them. |
| **Autonomous Skimmer Vessels** | A fleet of small autonomous vessels that actively steer towards plastic hotspots. |
| **System 03 (current)** | The existing system: one large barrier towed by two vessels, steered towards areas of high plastic concentration. |

Five stakeholder groups rate the alternatives: **The Ocean Cleanup**, **North Pacific commercial fishers**, **Investors/donors**, **Marine conservation advocates** and **Citizens**. These are the same five stakeholders as in the optimisation notebook `OceanCleanup_System03.ipynb`. Each stakeholder rates the alternatives on two to four criteria of their own and gives each criterion a weight that reflects how important it is to them.

The ratings are qualitative estimates by the project team, based on the characteristics and trade-offs found during the exploration of alternatives. They are not measured values.

This notebook combines the ratings into a single group preference score per alternative, so that the alternatives can be ranked and compared.

## Importing Required Packages

`numpy` and `pandas` are used for data handling and numerical operations.

The local module `a_fine_aggregator` performs the aggregation (see `genetic_algorithm_pfm/a_fine_aggregator/` for the implementation). It is imported relative to this notebook, so the notebook must be run from the `Our project` folder, which contains its own copy of `genetic_algorithm_pfm`. Running it from another folder gives `ModuleNotFoundError: No module named 'genetic_algorithm_pfm'`.

In [1]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Round the float values in the dataframe to 2 decimal places
pd.options.display.float_format = '{:.2f}'.format

# Import local module for a-fine-aggregator
from genetic_algorithm_pfm.a_fine_aggregator import a_fine_aggregator

## Loading the Ratings Data

The stakeholder ratings are stored in `MCDA_input/MCDA_Ratings_OceanCleanup_System03.csv`. Each row holds one stakeholder's ratings of the four alternatives on one criterion, together with the weight (`Criteria_Weight`) that the stakeholder gives to that criterion.

Each stakeholder uses their own set of criteria. Some criteria appear for more than one stakeholder, but with different ratings and weights, because stakeholders can value the same criterion differently. *Scalability*, for example, has a weight of 0.20 for The Ocean Cleanup and 0.35 for the investors/donors.

The list of `alternatives` is taken from the columns between `Criteria` and `Criteria_Weight`, and the list of `stakeholders` from the unique values in the `Stakeholder` column. Both lists are printed and the ratings table is displayed, to check that the data were read correctly.

In [2]:
ratings = pd.read_csv(
    "MCDA_input/MCDA_Ratings_OceanCleanup_System03.csv",
    sep=";",       # fields are semicolon-separated, not comma-separated
    skiprows=1     # skip the "MCDA The Ocean Cleanup Alternatives" title row
)

alternatives = list(ratings.columns[2:-1].unique()) # Get the list of alternatives from the dataframe columns, excluding the first two and last column
stakeholders = list(ratings["Stakeholder"].unique()) # Get the list of stakeholders from the "Stakeholder" column in the dataframe
print(f"Alternatives: {alternatives}")
print(f"Stakeholders: {stakeholders}")
display(ratings)


Alternatives: ['Modular Fleet', 'Anchored Barrier Array', 'Autonomous Skimmer Vessels', 'System 03 (current)']
Stakeholders: ['The Ocean Cleanup', 'North Pacific commercial fishers', 'Investors/donors', 'Marine conservation advocates', 'Citizens']


,Stakeholder,Criteria,Modular Fleet,Anchored Barrier Array,Autonomous Skimmer Vessels,System 03 (current),Criteria_Weight
0,The Ocean Cleanup,Plastic removal efficiency (kg per time),65,45,75,90,0.35
1,The Ocean Cleanup,Cost per kg removed,75,80,55,60,0.25
2,The Ocean Cleanup,Scalability,85,55,80,70,0.20
3,The Ocean Cleanup,Ecological/bycatch impact,70,75,65,65,0.20
4,North Pacific commercial fishers,Fishing ground access,60,40,80,65,0.40
5,North Pacific commercial fishers,Gear conflict / entanglement risk,55,40,80,65,0.35
6,North Pacific commercial fishers,Vessel traffic disruption,75,90,65,50,0.25
7,Investors/donors,Scalability,60,45,75,95,0.35
8,Investors/donors,Ecological impact,75,80,50,60,0.35
9,Investors/donors,Cost per kg removed,85,55,80,70,0.30


## Checking the Criteria Weights

For the a-fine aggregator to give a meaningful result, each stakeholder's criteria weights must sum to one (100%). This is checked here explicitly, because a mismatch would otherwise carry through to the final scores without any warning.

The loop below sums the `Criteria_Weight` values per stakeholder and marks any stakeholder whose weights do not sum to 1 (within a small numerical tolerance) as a `MISMATCH`.

In [3]:
# Check each stakeholder's weights sum to 1 (i.e. 100%)
print("Check weights per stakeholder:")
all_valid = True

for stakeholder in stakeholders:
    stakeholder_weights = ratings.loc[ratings["Stakeholder"] == stakeholder, "Criteria_Weight"]
    total = stakeholder_weights.sum()
    is_valid = np.isclose(total, 1)
    all_valid &= is_valid

    status = "OK" if is_valid else "MISMATCH"
    print(f"  {stakeholder:<40s}: {total:6.2f}  [{status}]")


Check weights per stakeholder:
  The Ocean Cleanup                       :   1.00  [OK]
  North Pacific commercial fishers        :   1.00  [OK]
  Investors/donors                        :   1.00  [OK]
  Marine conservation advocates           :   1.00  [OK]
  Citizens                                :   1.00  [OK]


## Stakeholder Weights

Just as each stakeholder weighs their own criteria, the five stakeholder groups must be weighted relative to one another before their scores can be combined into one group score.

$$\sum_{i=1}^{5} w_i = 1$$

The weights are given in the order in which the stakeholders appear in the `.csv` file:

| Weight | Stakeholder | `weights_eq` | `weights_dom` |
|---|---|---|---|
| $w_1$ | The Ocean Cleanup | 0.20 | 0.40 |
| $w_2$ | North Pacific commercial fishers | 0.20 | 0.20 |
| $w_3$ | Investors/donors | 0.20 | 0.20 |
| $w_4$ | Marine conservation advocates | 0.20 | 0.10 |
| $w_5$ | Citizens | 0.20 | 0.10 |

- `weights_eq` gives every stakeholder the same weight, so no single stakeholder dominates the outcome.
- `weights_dom` gives The Ocean Cleanup, as the owner and operator of the system, twice the weight of the fishers and the investors/donors, and four times the weight of the conservation advocates and the citizens.

The variable `stakeholder_weights` selects which set is used. Note that the order differs from the optimisation notebook `OceanCleanup_System03.ipynb`, where the investors/donors come first.

In [4]:
# Set stakeholder weights
# Order:        Ocean Cleanup, fishers, investors/donors, conservation advocates, citizens
weights_eq =    [0.2, 0.2, 0.2, 0.2, 0.2]  # equal weights for stakeholders
weights_dom =   [0.4, 0.2, 0.2, 0.10, 0.10]  # The Ocean Cleanup dominant

stakeholder_weights = weights_eq

assert np.isclose(sum(stakeholder_weights), 1), f"Weights must sum to 1, got {sum(stakeholder_weights)}"


## Aggregating the Preference Scores

The group preference score per alternative is calculated with the `a_fine_aggregator`, in **two levels**:

1. **Level 1 — criteria → stakeholder score.** For each stakeholder, the ratings of the four alternatives on each criterion are combined using that stakeholder's own criteria weights. This gives one preference score per stakeholder per alternative.
2. **Level 2 — stakeholders → group score.** The stakeholder scores (one row per stakeholder, one column per alternative) are combined a second time, now using the `stakeholder_weights` defined above. This gives one final preference score per alternative.

The argument `scores_range=(-0.0, -100.0)` rescales the aggregated scores so that the least preferred alternative gets **0** and the most preferred alternative gets **100**. The negative range is intentional: the aggregator is written for minimisation problems and therefore works with negative preference values.

In [5]:
# Calculate the aggregated scores for each alternative using the a-fine-aggregator
# --- Level 1: aggregate criteria ratings -> one score per stakeholder per alternative ---
stakeholder_scores = {}

for stakeholder in stakeholders:
    stakeholder_data = ratings.loc[ratings["Stakeholder"] == stakeholder]
    criteria_weights = stakeholder_data["Criteria_Weight"].to_numpy()
    p = stakeholder_data[alternatives].to_numpy()  # shape: n_criteria x n_alternatives
    stakeholder_scores[stakeholder] = a_fine_aggregator(criteria_weights, p, scores_range=(-0.0, -100.0))

# Collect into matrix: rows = stakeholders, columns = alternatives (order matches `alternatives`)
stakeholder_score_matrix = np.array([stakeholder_scores[s] for s in stakeholders])

print("Individual stakeholder aggregated scores:")
display(pd.DataFrame(stakeholder_score_matrix, index=stakeholders, columns=alternatives))

# --- Level 2: aggregate stakeholder scores -> final preference score per alternative ---
final_scores = a_fine_aggregator(stakeholder_weights, stakeholder_score_matrix, scores_range=(-0.0, -100.0))

results = (
    pd.DataFrame(final_scores, index=alternatives, columns=["Preference score"])
    .round(2)
    .sort_values("Preference score", ascending=False)
)

print("Final aggregated preference scores per alternative:")
display(results)


Individual stakeholder aggregated scores:


,Modular Fleet,Anchored Barrier Array,Autonomous Skimmer Vessels,System 03 (current)
The Ocean Cleanup,100.00,1.66,0.00,44.69
North Pacific commercial fishers,40.13,0.00,100.00,37.16
Investors/donors,100.00,0.00,37.04,81.48
Marine conservation advocates,72.54,100.00,0.00,90.18
Citizens,41.73,0.00,60.62,100.00


Final aggregated preference scores per alternative:


,Preference score
System 03 (current),100.00
Modular Fleet,97.70
Autonomous Skimmer Vessels,42.21
Anchored Barrier Array,0.00


## Interpreting the Results

The `results` table ranks the four alternatives by their group preference score. Because the aggregator rescales the scores at both levels, the worst alternative always gets 0 and the best always gets 100. The scores therefore show how the alternatives stand relative to each other; they are not an absolute measure of quality. A score of 0 for the Anchored Barrier Array means it is the least preferred of these four, not that it has no value.

With equal stakeholder weights (`weights_eq`):

| Rank | Alternative | Preference score |
|---|---|---|
| 1 | System 03 (current) | 100.00 |
| 2 | Modular Fleet | 97.70 |
| 3 | Autonomous Skimmer Vessels | 42.21 |
| 4 | Anchored Barrier Array | 0.00 |

- **System 03 and the Modular Fleet are almost tied**, well ahead of the other two alternatives.
- The **Anchored Barrier Array** is last for the fishers, the investors/donors and the citizens, and nearly last for The Ocean Cleanup (1.66). It cannot follow the plastic and it permanently occupies fishing grounds. Only the conservation advocates rank it first, because it has the highest rating on their heaviest criterion, *Unintended ecological harm (bycatch risk)*.
- The **Autonomous Skimmer Vessels** are clearly the first choice of the fishers, because small mobile vessels can avoid fishing operations, but they are last for The Ocean Cleanup and the conservation advocates.
- The stakeholders disagree on the top two: The Ocean Cleanup and the investors/donors prefer the Modular Fleet, while the citizens prefer System 03 and the conservation advocates rank System 03 (90.18) well above the Modular Fleet (72.54).

The ranking of the top two depends on the stakeholder weights. With `weights_dom`, where The Ocean Cleanup has a weight of 0.40, the Modular Fleet comes first (100.00) and System 03 second (74.41); the Autonomous Skimmer Vessels (36.91) and the Anchored Barrier Array (0.00) keep their places. The conclusion that holds for both weight sets is that the Anchored Barrier Array and the Autonomous Skimmer Vessels are less preferred than System 03 and the Modular Fleet. Which of those two is preferred depends on whose interests weigh most.

To test this further, change `stakeholder_weights` above, or the `Criteria_Weight` values and ratings in the `.csv` file, and run the notebook again.